# Land-AI DEM 타일 재생성 (전국 위도 33~39°)

**실행 방법**: 상단 메뉴 → **런타임(Runtime) → 모두 실행(Run all)** 또는 `Ctrl+F9`

그 다음 탭을 열어두고 기다리면 됩니다.  
이미 완료된 타일은 자동으로 건너뜁니다 (약 2~5시간 예상).

In [ ]:
!pip install boto3 scipy Pillow numpy -q
print("패키지 설치 완료")

In [ ]:
%%writefile retile_dem.py
"""
0.1° 등고선 타일 → 0.01° DEM PNG 직접 변환
★ v2: 타일 키 범위가 아닌 실제 포인트 좌표 기반으로 서브타일 생성

사용법:
  python retile_dem.py <lat_start> <lat_end>
  예: python retile_dem.py 33 35
"""
import boto3, gzip, json, io, os, sys, time, math
import numpy as np
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

R2_ENDPOINT   = "https://1f86bdf44ba6c9801d5064618be271d7.r2.cloudflarestorage.com"
R2_ACCESS_KEY = os.environ.get("R2_ACCESS_KEY", "a37b8798d6486a748b280866d2b01413")
R2_SECRET_KEY = os.environ.get("R2_SECRET_KEY", "e7553f8bee37a90463ee15fd6ad65db16f0ada7b3f3b113368aa30869ae5c8fb")
R2_BUCKET     = "landai-projects"

DEM_SIZE = 256
WORKERS  = 8

client = boto3.client(
    "s3", endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY, aws_secret_access_key=R2_SECRET_KEY,
    config=Config(retries={"max_attempts": 5, "mode": "adaptive"},
                  connect_timeout=30, read_timeout=60),
)


def to_terrarium_png(elev_grid: np.ndarray) -> bytes:
    filled = np.where(np.isnan(elev_grid), 0.0, elev_grid)
    filled = np.clip(filled, -10000.0, 20000.0)
    enc = np.clip(np.round((filled + 32768.0) * 256.0).astype(np.uint32), 0, 0xFFFFFF)
    r = ((enc >> 16) & 0xFF).astype(np.uint8)
    g = ((enc >>  8) & 0xFF).astype(np.uint8)
    b = ( enc        & 0xFF).astype(np.uint8)
    img = Image.fromarray(np.stack([r, g, b], axis=-1), "RGB")
    buf = io.BytesIO()
    img.save(buf, "PNG", optimize=False)
    return buf.getvalue()


def process_source_tile(src_key: str) -> tuple:
    """0.1° 등고선 타일 1개 → 실제 좌표 기반 0.01° DEM PNG 업로드"""
    try:
        resp = client.get_object(Bucket=R2_BUCKET, Key=src_key)
        body = resp["Body"].read()
        if body[:2] == b'\x1f\x8b':
            body = gzip.decompress(body)
        features = json.loads(body).get("features", [])
    except Exception as e:
        return (src_key, "ERR", f"download: {e}")

    # 모든 포인트 수집
    all_pts = []
    for feat in features:
        props = feat.get("properties", {})
        elev  = props.get("e")
        if elev is None:
            continue
        try:
            elev = float(elev)
        except (ValueError, TypeError):
            continue
        geom   = feat.get("geometry", {})
        gtype  = geom.get("type", "")
        coords = geom.get("coordinates", [])
        if gtype == "LineString":
            for c in coords:
                all_pts.append((c[1], c[0], elev))
        elif gtype == "MultiLineString":
            for line in coords:
                for c in line:
                    all_pts.append((c[1], c[0], elev))

    if len(all_pts) < 4:
        return (src_key, "SKIP", f"points={len(all_pts)}")

    # ★ 타일 키 범위 대신 실제 포인트 좌표 범위로 서브타일 결정
    lats_all = [p[0] for p in all_pts]
    lons_all = [p[1] for p in all_pts]
    lat_idx_min = math.floor(min(lats_all) * 100)
    lat_idx_max = math.floor(max(lats_all) * 100)
    lon_idx_min = math.floor(min(lons_all) * 100)
    lon_idx_max = math.floor(max(lons_all) * 100)

    ok_count = skip_count = 0

    for lat_idx in range(lat_idx_min, lat_idx_max + 1):
        for lon_idx in range(lon_idx_min, lon_idx_max + 1):
            sub_lat0 = lat_idx / 100.0
            sub_lon0 = lon_idx / 100.0
            sub_lat1 = (lat_idx + 1) / 100.0
            sub_lon1 = (lon_idx + 1) / 100.0

            sub_pts = [
                p for p in all_pts
                if sub_lat0 <= p[0] < sub_lat1 and sub_lon0 <= p[1] < sub_lon1
            ]
            if len(sub_pts) < 4:
                skip_count += 1
                continue

            dst_key = f"dem/{lat_idx}_{lon_idx}.png"

            try:
                client.head_object(Bucket=R2_BUCKET, Key=dst_key)
                skip_count += 1
                continue
            except Exception:
                pass

            try:
                lats  = np.array([p[0] for p in sub_pts])
                lons  = np.array([p[1] for p in sub_pts])
                elevs = np.array([p[2] for p in sub_pts])

                if len(sub_pts) > 8000:
                    idx = np.random.choice(len(sub_pts), 8000, replace=False)
                    lats, lons, elevs = lats[idx], lons[idx], elevs[idx]

                glat_1d = np.linspace(sub_lat1, sub_lat0, DEM_SIZE)
                glon_1d = np.linspace(sub_lon0, sub_lon1, DEM_SIZE)
                glon, glat = np.meshgrid(glon_1d, glat_1d)

                interp = LinearNDInterpolator(list(zip(lats, lons)), elevs, fill_value=float("nan"))
                elev_grid = interp(glat, glon).astype(np.float32)

                nan_mask = np.isnan(elev_grid)
                if nan_mask.any():
                    nn = NearestNDInterpolator(list(zip(lats, lons)), elevs)
                    elev_grid = np.where(nan_mask, nn(glat, glon).astype(np.float32), elev_grid)

                png_bytes = to_terrarium_png(elev_grid)
                client.put_object(Bucket=R2_BUCKET, Key=dst_key,
                                  Body=png_bytes, ContentType="image/png")
                ok_count += 1
            except Exception:
                skip_count += 1

    return (src_key, "OK", f"uploaded={ok_count} skip={skip_count}")


if __name__ == "__main__":
    if len(sys.argv) < 3:
        print("Usage: python retile_dem.py <lat_start> <lat_end>")
        sys.exit(1)

    lat_start = float(sys.argv[1])
    lat_end   = float(sys.argv[2])

    print(f"R2 타일 목록 조회 중... (위도 {lat_start}~{lat_end}°)")
    keys = []
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=R2_BUCKET, Prefix="contours/"):
        for obj in page.get("Contents", []):
            k = obj["Key"]
            name = k.split("/")[-1].split(".")[0]
            parts = name.split("_")
            if len(parts[0]) != 3:
                continue
            lat_k = int(parts[0]) / 10.0
            if lat_start <= lat_k < lat_end:
                keys.append(k)

    print(f"대상 타일: {len(keys)}개 (위도 {lat_start}~{lat_end}°)")
    if not keys:
        print("처리할 타일 없음.")
        sys.exit(0)

    ok = err = skip = done = 0
    start = time.time()

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for src, status, msg in ex.map(process_source_tile, keys):
            done += 1
            if status == "OK":
                ok += 1
            elif status == "ERR":
                err += 1
                print(f"[ERR] {src}: {msg}")
            else:
                skip += 1

            if done % 50 == 0 or done == len(keys):
                elapsed = time.time() - start
                eta = elapsed / done * (len(keys) - done) if done < len(keys) else 0
                print(f"[{done}/{len(keys)}] OK:{ok} SKIP:{skip} ERR:{err} "
                      f"속도:{done/elapsed:.1f}/s ETA:{eta/60:.0f}분")

    elapsed = time.time() - start
    print(f"\n=== 완료 ===")
    print(f"위도 {lat_start}~{lat_end}° | OK:{ok} SKIP:{skip} ERR:{err} | {elapsed/60:.1f}분")


In [ ]:
import subprocess, sys, time

ranges = [(33, 35), (35, 36), (36, 37), (37, 38), (38, 39)]
total_start = time.time()

for lat_s, lat_e in ranges:
    print(f"
{'='*60}")
    print(f"위도 {lat_s}~{lat_e}도 처리 중...")
    print(f"{'='*60}")
    t = time.time()
    r = subprocess.run([sys.executable, "retile_dem.py", str(lat_s), str(lat_e)])
    elapsed = time.time() - t
    status = "완료" if r.returncode == 0 else "실패"
    print(f"
[{status}] 소요: {elapsed/60:.1f}분")

total = time.time() - total_start
print(f"
{'='*60}")
print(f"전체 완료! 총 소요: {total/60:.1f}분")